## Calibration of document parsing pipeline for data extraction

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import os
import os.path as osp
from pathlib import Path
import sys

root = Path.cwd().parent 
if str(root) not in sys.path:
    sys.path.append(str(root))

In [3]:
from src.config import CFG
from src.utils import normalize_text

In [4]:
import pandas as pd
import pdfplumber

import re

In [5]:
# CFG.PROJECT_ROOT, CFG.DATA_DIR, CFG.PROCESSED_DATA_DIR

In [6]:
raw_data_path = osp.join(CFG.DATA_DIR, "raw")

pdf_path = osp.join(raw_data_path, os.listdir(raw_data_path)[0])

pdf_path

'/root/chat-app/data/raw/EDAN_2025_RESULTAT_NATIONAL_DETAILS.pdf'

## Data Extraction Pipeline

In [7]:
def get_data_from_pdf(
        pdf_path:str
    ):
    all_tables      = []
    all_tables_df   = []

    pdf_cols = [
        'REGION',
        'CONSTITUENCY_NUM',
        'CONSTITUENCY_TITLE',
        'NUM_POLLING_STATIONS',
        'REGISTERED',
        'NUM_VOTERS',
        'PART_RATE',
        'NULL_BALL',
        'EXPRESSED_VOTES',
        'NUM_BLANK',
        'PCT_BLANK',
        'PARTY_NAME',
        'CANDIDATE_NAME',
        'NUM_VOTES',
        'PCT_SCORE',
        'IS_WINNER',
        'PAGE_ID'
    ]
    
    settings        = {
        "vertical_strategy": "lines",
        "horizontal_strategy": "lines", 
        "snap_tolerance": 4,            
        "join_tolerance": 4,            
        "intersection_tolerance": 10, 
    }
    
    with pdfplumber.open(pdf_path) as pdf:
        page_id = 1
        for page in pdf.pages:
            tables = page.extract_tables(table_settings=settings)
            for table in tables:
                all_tables.append(table)
                df            = pd.DataFrame(table[2:], columns=table[:2])
                df["PAGE_ID"] = [page_id]*df.shape[0]                
                all_tables_df.append(df)
            page_id+=1
    
    try:
        for df in all_tables_df:
            df.columns = pdf_cols
            
        final_df            = pd.concat(all_tables_df, axis=0, ignore_index=True)
        totals              = final_df.iloc[0] # extract totals line from final table
        final_df            = final_df.drop(index=0).reset_index(drop=True)

        # fix region names
        final_df["REGION"]  = final_df["REGION"].replace(r'^\s*$', np.nan, regex=True) # making sure region names to not come out empty
        final_df["REGION"]  = final_df["REGION"].apply(lambda r: clean_region_name(r))
        final_df            = final_df.ffill()

        return totals, final_df, all_tables, all_tables_df
    
    except Exception as e:
        print(f" Error while merging data: {e}")
        return None, None, all_tables, all_tables_df
    
def clean_region_name(region:str):
    if region not in [None, np.nan] and "\n" in region:
        return region.replace("\n", "")[::-1]
    else:
        return region
    
def summarize_turnout(totals:pd.Series):
    turnout_summary = "These are the overall numbers from the 2025 legislative elections: "

    for k in totals.keys()[1:]:
        if totals[k] not in [None, ''] and totals[k]:
            entry=f"{k}={totals[k]}, "
            turnout_summary+=entry

    turnout_summary = turnout_summary.strip(' ,')

    return turnout_summary

In [8]:
totals, df, tables, tables_df = get_data_from_pdf(pdf_path)

if df is not None:
    print(df.shape)

(1124, 17)


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1124 entries, 0 to 1123
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   REGION                1124 non-null   object
 1   CONSTITUENCY_NUM      1124 non-null   object
 2   CONSTITUENCY_TITLE    1124 non-null   object
 3   NUM_POLLING_STATIONS  1124 non-null   object
 4   REGISTERED            1124 non-null   object
 5   NUM_VOTERS            1124 non-null   object
 6   PART_RATE             1124 non-null   object
 7   NULL_BALL             1124 non-null   object
 8   EXPRESSED_VOTES       1124 non-null   object
 9   NUM_BLANK             1124 non-null   object
 10  PCT_BLANK             1124 non-null   object
 11  PARTY_NAME            1124 non-null   object
 12  CANDIDATE_NAME        1124 non-null   object
 13  NUM_VOTES             1124 non-null   object
 14  PCT_SCORE             1124 non-null   object
 15  IS_WINNER             1124 non-null   

In [10]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
PAGE_ID,1124.0,17.951068,9.967369,1.0,9.0,18.0,27.0,35.0


In [11]:
def get_avg_vals(df:pd.DataFrame, col_name:str, dtype:str='int'):

    if dtype=='int':
        l = [int(n.replace(' ', '')) for n in df[col_name] if n!='']
        avg = sum(l)//len(l)
    else:
        l = [float(pct.replace('%', '').replace(',', '.')) for pct in df[col_name]]
        avg = sum(l)/len(l)
    
    return avg



In [12]:
get_avg_vals(df, 'NUM_POLLING_STATIONS'), get_avg_vals(df, 'PART_RATE', dtype='float')

(127, 37.19553380782918)

In [13]:
num_cols = [
    'NUM_POLLING_STATIONS',
    'REGISTERED',
    'NUM_VOTERS',
    'NULL_BALL',
    'EXPRESSED_VOTES',
    'NUM_BLANK',
    'NUM_VOTES',
]

pct_cols = [
    'PART_RATE',
    'PCT_BLANK',
    'PCT_SCORE',
]

# process int-based columns
for col in num_cols:
    df[col] = df[col].apply(lambda n: n.replace(' ', ''))
    df[col] = df[col].replace('', get_avg_vals(df, col, 'int'))

# process float-based columns
for col in pct_cols:
    df[col] = df[col].apply(lambda n: n.replace('%', '').replace(',', '.'))
    df[col] = df[col].replace('', get_avg_vals(df, col, 'float'))

In [14]:
df.head(2)

,REGION,CONSTITUENCY_NUM,CONSTITUENCY_TITLE,NUM_POLLING_STATIONS,REGISTERED,NUM_VOTERS,PART_RATE,NULL_BALL,EXPRESSED_VOTES,NUM_BLANK,PCT_BLANK,PARTY_NAME,CANDIDATE_NAME,NUM_VOTES,PCT_SCORE,IS_WINNER,PAGE_ID
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,KOTO EHOU SOPIE,547,4.00,,1
1,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,RHDP,KOFFI AKA CHARLES,9078,66.35,ELU(E),1


In [15]:
df = df.astype({
    'REGION':str,
    'CONSTITUENCY_NUM':str,
    'CONSTITUENCY_TITLE':str,
    'NUM_POLLING_STATIONS':int,
    'REGISTERED':int,
    'NUM_VOTERS':int,
    'PART_RATE':float,
    'NULL_BALL':int,
    'EXPRESSED_VOTES':int,
    'NUM_BLANK':int,
    'PCT_BLANK':float,
    'PARTY_NAME':str,
    'CANDIDATE_NAME':str,
    'NUM_VOTES':int,
    'PCT_SCORE':float,
    'IS_WINNER':bool,
    'PAGE_ID': int
})

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1124 entries, 0 to 1123
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   REGION                1124 non-null   object 
 1   CONSTITUENCY_NUM      1124 non-null   object 
 2   CONSTITUENCY_TITLE    1124 non-null   object 
 3   NUM_POLLING_STATIONS  1124 non-null   int64  
 4   REGISTERED            1124 non-null   int64  
 5   NUM_VOTERS            1124 non-null   int64  
 6   PART_RATE             1124 non-null   float64
 7   NULL_BALL             1124 non-null   int64  
 8   EXPRESSED_VOTES       1124 non-null   int64  
 9   NUM_BLANK             1124 non-null   int64  
 10  PCT_BLANK             1124 non-null   float64
 11  PARTY_NAME            1124 non-null   object 
 12  CANDIDATE_NAME        1124 non-null   object 
 13  NUM_VOTES             1124 non-null   int64  
 14  PCT_SCORE             1124 non-null   float64
 15  IS_WINNER            

In [17]:
df.head(10)

,REGION,CONSTITUENCY_NUM,CONSTITUENCY_TITLE,NUM_POLLING_STATIONS,REGISTERED,NUM_VOTERS,PART_RATE,NULL_BALL,EXPRESSED_VOTES,NUM_BLANK,PCT_BLANK,PARTY_NAME,CANDIDATE_NAME,NUM_VOTES,PCT_SCORE,IS_WINNER,PAGE_ID
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,KOTO EHOU SOPIE,547,4.00,False,1
1,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,RHDP,KOFFI AKA CHARLES,9078,66.35,True,1
2,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,YEPIE KOUASSI THEODORE,58,0.42,False,1
3,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,TCHIMOU GNAMON BERTRAND,1991,14.55,False,1
4,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,INDEPENDANT,BOKA BOKA MAURICE,674,4.93,False,1
5,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,ADCI,EDI DOFFOU PAUL,331,2.42,False,1
6,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,FPI,N'GUESSAN KOTCHI REMI,474,3.46,False,1
7,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,MGC,NANDO YAVO SERGE,453,3.31,False,1
8,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,133,48710,12821,26.32,317,12504,81,0.65,ADCI,OCHOU WROHOUM MARIE-PASCALE,296,2.37,False,1
9,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,133,48710,12821,26.32,317,12504,81,0.65,RHDP,DIMBA N'GOU PIERRE,10675,85.37,True,1


### Save procesed dataframe

In [18]:
df.to_csv(osp.join(CFG.PROCESSED_DATA_DIR, '2025_legislative_elections.csv'), index=False)

### Turnout summary from totals

In [19]:
summarize_turnout(totals)

'These are the overall numbers from the 2025 legislative elections: NUM_POLLING_STATIONS=25 338, REGISTERED=8 597 092, NUM_VOTERS=3 012 094, PART_RATE=35,04%, NULL_BALL=68 525, EXPRESSED_VOTES=2 943 569, NUM_BLANK=29 578, PCT_BLANK=1,00%, NUM_VOTES=2 913 991, PAGE_ID=1'

### Grouping

In [20]:
# Group by everything that identifies the location
group_cols = ['REGION', 'CONSTITUENCY_NUM', 'CONSTITUENCY_TITLE', 'NUM_POLLING_STATIONS', 'REGISTERED', 'NUM_VOTERS',
              'PART_RATE', 'NULL_BALL', 'EXPRESSED_VOTES', 'NUM_BLANK', 'PCT_BLANK', 'PAGE_ID']

df_grouped = df.groupby(group_cols).agg({
    'PARTY_NAME': list,
    'CANDIDATE_NAME': list,
    'NUM_VOTES': list,
    'PCT_SCORE': list,
    'IS_WINNER': 'first'
}).reset_index()


In [21]:
df_grouped

,REGION,CONSTITUENCY_NUM,CONSTITUENCY_TITLE,NUM_POLLING_STATIONS,REGISTERED,NUM_VOTERS,PART_RATE,NULL_BALL,EXPRESSED_VOTES,NUM_BLANK,PCT_BLANK,PAGE_ID,PARTY_NAME,CANDIDATE_NAME,NUM_VOTES,PCT_SCORE,IS_WINNER
0,AGNEBY-TIASSA,,,127,31758,9221,29.04,283,8938,72,0.81,2,[RHDP],[KANGBE YAYORO CHARLES LOPEZ],[3169],[35.46],True
1,AGNEBY-TIASSA,,,127,39362,16841,42.78,518,16323,118,0.72,2,[PDCI-RDA],[MIAN OCTAVI PEROU],[159],[0.97],False
2,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,1,"[INDEPENDANT, RHDP, INDEPENDANT, INDEPENDANT, ...","[KOTO EHOU SOPIE, KOFFI AKA CHARLES, YEPIE KOU...","[547, 9078, 58, 1991, 674, 331, 474, 453]","[4.0, 66.35, 0.42, 14.55, 4.93, 2.42, 3.46, 3.31]",False
3,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,133,48710,12821,26.32,317,12504,81,0.65,1,"[ADCI, RHDP, INDEPENDANT, INDEPENDANT, INDEPEN...","[OCHOU WROHOUM MARIE-PASCALE, DIMBA N'GOU PIER...","[296, 10675, 20, 75, 30, 1327]","[2.37, 85.37, 0.16, 0.6, 0.24, 10.61]",False
4,AGNEBY-TIASSA,003,AZAGUIE COMMUNE ET SOUS-PREFECTURE,44,15515,5174,33.35,73,5101,24,0.47,1,"[INDEPENDANT, INDEPENDANT, INDEPENDANT, INDEPE...","[KOUAME YAO FREDERIC, DOUMBIA HAMED CHEICK YVA...","[439, 1549, 863, 38, 96, 1673, 256, 163]","[8.61, 30.37, 16.92, 0.74, 1.88, 32.8, 5.02, 3.2]",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
212,WORODOUGOU,201,"KANI, COMMUNE ET SOUS-PREFECTURE",41,13143,7621,57.99,143,7478,115,1.54,34,"[INDEPENDANT, INDEPENDANT, RHDP, INDEPENDANT]","[YACOUBA MEITE, MEITE YAYA, MEITE BEN ABDOULAY...","[202, 4329, 2530, 302]","[2.7, 57.89, 33.83, 4.04]",False
213,WORODOUGOU,202,"BOBI-DIARABANA, COMMUNE ET SOUS-PREFECTURE,\nS...",60,16811,9578,56.97,69,9509,114,1.20,34,"[RHDP, INDEPENDANT, INDEPENDANT]","[DIOMANDE MAMADOU, FOFANA VATIECOUMBA, FOFANA ...","[6519, 2567, 309]","[68.56, 27.0, 3.25]",True
214,WORODOUGOU,203,"SEGUELA, COMMUNE",55,20148,7201,35.74,80,7121,171,2.40,34,"[INDEPENDANT, INDEPENDANT, RHDP, INDEPENDANT, ...","[SOUMAHORO ADAMA, MOUSTAPHA FALL, TIMITE SAHIN...","[357, 1214, 3038, 123, 2023, 195]","[5.01, 17.05, 42.66, 1.73, 28.41, 2.74]",False
215,WORODOUGOU,204,"DUALLA ET MASSALA, COMMUNES ET SOUS-\nPREFECTURES",48,12876,7334,56.96,85,7249,56,0.77,35,"[INDEPENDANT, INDEPENDANT, RHDP, INDEPENDANT, ...","[DOSSO ABOUBACAR SIDIKY, COULIBALY METOBA, DOS...","[2392, 2076, 2545, 7, 173]","[33.0, 28.64, 35.11, 0.1, 2.39]",False


In [22]:
df_grouped.shape

(217, 17)

### Turnout data

In [23]:
turnout_data = df[group_cols].drop_duplicates()

turnout_data

,REGION,CONSTITUENCY_NUM,CONSTITUENCY_TITLE,NUM_POLLING_STATIONS,REGISTERED,NUM_VOTERS,PART_RATE,NULL_BALL,EXPRESSED_VOTES,NUM_BLANK,PCT_BLANK,PAGE_ID
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",144,52106,14070,27.00,388,13682,76,0.56,1
8,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,133,48710,12821,26.32,317,12504,81,0.65,1
14,AGNEBY-TIASSA,003,AZAGUIE COMMUNE ET SOUS-PREFECTURE,44,15515,5174,33.35,73,5101,24,0.47,1
22,AGNEBY-TIASSA,004,"ANANGUIE, CECHI ET RUBINO, COMMUNES ET SOUS-\n...",72,23466,7650,32.60,241,7409,49,0.66,1
32,AGNEBY-TIASSA,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES",102,37720,10768,28.55,347,10421,134,1.29,2
...,...,...,...,...,...,...,...,...,...,...,...,...
1101,WORODOUGOU,201,"KANI, COMMUNE ET SOUS-PREFECTURE",41,13143,7621,57.99,143,7478,115,1.54,34
1105,WORODOUGOU,202,"BOBI-DIARABANA, COMMUNE ET SOUS-PREFECTURE,\nS...",60,16811,9578,56.97,69,9509,114,1.20,34
1108,WORODOUGOU,203,"SEGUELA, COMMUNE",55,20148,7201,35.74,80,7121,171,2.40,34
1114,WORODOUGOU,204,"DUALLA ET MASSALA, COMMUNES ET SOUS-\nPREFECTURES",48,12876,7334,56.96,85,7249,56,0.77,35


In [24]:
turnout_data.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "turnout.parquet"), index=False)

### Results data

In [25]:
results_data = df[
    ['REGION', 'CONSTITUENCY_TITLE', 'CONSTITUENCY_NUM', 'PARTY_NAME', 
     'CANDIDATE_NAME', 'NUM_VOTES', 'PCT_SCORE', 'IS_WINNER', 'PAGE_ID']
]

results_data

,REGION,CONSTITUENCY_TITLE,CONSTITUENCY_NUM,PARTY_NAME,CANDIDATE_NAME,NUM_VOTES,PCT_SCORE,IS_WINNER,PAGE_ID
0,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,KOTO EHOU SOPIE,547,4.00,False,1
1,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,RHDP,KOFFI AKA CHARLES,9078,66.35,True,1
2,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,YEPIE KOUASSI THEODORE,58,0.42,False,1
3,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,TCHIMOU GNAMON BERTRAND,1991,14.55,False,1
4,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,BOKA BOKA MAURICE,674,4.93,False,1
...,...,...,...,...,...,...,...,...,...
1119,WORODOUGOU,"KAMALO, SIFIE ET WOROFLA, COMMUNES ET SOUS-\nP...",205,RHDP,MEITE ABOULAYE,11233,88.64,True,35
1120,WORODOUGOU,"KAMALO, SIFIE ET WOROFLA, COMMUNES ET SOUS-\nP...",205,INDEPENDANT,KONE LOSSENI,46,0.36,False,35
1121,WORODOUGOU,"KAMALO, SIFIE ET WOROFLA, COMMUNES ET SOUS-\nP...",205,INDEPENDANT,TRAORE SEKOU,646,5.10,False,35
1122,WORODOUGOU,"KAMALO, SIFIE ET WOROFLA, COMMUNES ET SOUS-\nP...",205,RDP,FOFANA SIAKA,175,1.38,False,35


In [26]:
results_data.PARTY_NAME.unique()

array(['INDEPENDANT', 'RHDP', 'ADCI', 'FPI', 'MGC', 'PDCI-RDA', 'CNPCIN',
       'ADCI - GP-PAIX - VALEUR', 'GJPA-CI', 'CNJB-ADO', 'PRO CI', 'CODE',
       'GP-PAIX', 'AIDE', 'PIA/PRI/CODE', 'UNCI', 'PDCI - FPI - ADCI',
       'MERCI', 'URCI', 'LE BUFFLE', 'UDCY', 'EDS', 'CRI PANAFRICAIN',
       'MDR', 'ICON', 'P.B.J.V', 'UNPR', 'REEL.CI', 'APR.CI', 'AIRD',
       'PPSD', 'ACI', 'MNRP', 'FPP', 'FNDR', 'UFD', 'MLPCI', 'CNDCI',
       'UDP', 'PIA/CODE', 'PDCI-RDA - EDS', 'MIRDEP', 'RDP'], dtype=object)

In [27]:
# results_data["IS_WINNER"] = results_data["IS_WINNER"].apply(lambda r: int(r=="ELU(E)"))
results_data = results_data.drop_duplicates()
results_data.head()

,REGION,CONSTITUENCY_TITLE,CONSTITUENCY_NUM,PARTY_NAME,CANDIDATE_NAME,NUM_VOTES,PCT_SCORE,IS_WINNER,PAGE_ID
0,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,KOTO EHOU SOPIE,547,4.00,False,1
1,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,RHDP,KOFFI AKA CHARLES,9078,66.35,True,1
2,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,YEPIE KOUASSI THEODORE,58,0.42,False,1
3,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,TCHIMOU GNAMON BERTRAND,1991,14.55,False,1
4,AGNEBY-TIASSA,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO...",001,INDEPENDANT,BOKA BOKA MAURICE,674,4.93,False,1


In [28]:
results_data.PARTY_NAME.unique()

array(['INDEPENDANT', 'RHDP', 'ADCI', 'FPI', 'MGC', 'PDCI-RDA', 'CNPCIN',
       'ADCI - GP-PAIX - VALEUR', 'GJPA-CI', 'CNJB-ADO', 'PRO CI', 'CODE',
       'GP-PAIX', 'AIDE', 'PIA/PRI/CODE', 'UNCI', 'PDCI - FPI - ADCI',
       'MERCI', 'URCI', 'LE BUFFLE', 'UDCY', 'EDS', 'CRI PANAFRICAIN',
       'MDR', 'ICON', 'P.B.J.V', 'UNPR', 'REEL.CI', 'APR.CI', 'AIRD',
       'PPSD', 'ACI', 'MNRP', 'FPP', 'FNDR', 'UFD', 'MLPCI', 'CNDCI',
       'UDP', 'PIA/CODE', 'PDCI-RDA - EDS', 'MIRDEP', 'RDP'], dtype=object)

In [29]:
results_data[results_data["IS_WINNER"]==1].PARTY_NAME.unique()

array(['RHDP', 'INDEPENDANT', 'PDCI-RDA', 'LE BUFFLE', 'UNPR', 'FPI'],
      dtype=object)

In [30]:
results_data.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "results.parquet"), index=False)

### Region data

In [31]:
regions = pd.DataFrame(df["REGION"].unique(), columns=["REGION_NAME"]).drop_duplicates()

regions

,REGION_NAME
0,AGNEBY-TIASSA
1,BAFING
2,BAGOUE
3,BELIER
4,BERE
5,BOUNKANI
6,CAVALLY
7,DISTRICTAUTONOMED'ABIDJAN
8,DISTRICTAUTONOMEDEYAMOUSSOUKRO
9,FOLON


In [32]:
regions.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "regions.parquet"), index=False)

### Constituency

In [33]:
circs = df[
    ["REGION", "CONSTITUENCY_NUM", "CONSTITUENCY_TITLE"]
].drop_duplicates().reset_index(drop=True)

circs

,REGION,CONSTITUENCY_NUM,CONSTITUENCY_TITLE
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,\nLO..."
1,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE
2,AGNEBY-TIASSA,003,AZAGUIE COMMUNE ET SOUS-PREFECTURE
3,AGNEBY-TIASSA,004,"ANANGUIE, CECHI ET RUBINO, COMMUNES ET SOUS-\n..."
4,AGNEBY-TIASSA,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES"
...,...,...,...
207,WORODOUGOU,201,"KANI, COMMUNE ET SOUS-PREFECTURE"
208,WORODOUGOU,202,"BOBI-DIARABANA, COMMUNE ET SOUS-PREFECTURE,\nS..."
209,WORODOUGOU,203,"SEGUELA, COMMUNE"
210,WORODOUGOU,204,"DUALLA ET MASSALA, COMMUNES ET SOUS-\nPREFECTURES"


In [34]:
circs["CONSTITUENCY_TITLE"] = circs["CONSTITUENCY_TITLE"].apply(lambda c: c.replace('-\n', '-').replace('\n', '').strip())
circs["LEN_CIRC"] = circs["CONSTITUENCY_TITLE"].apply(lambda c: len(c))

circs = circs.drop_duplicates()

circs

,REGION,CONSTITUENCY_NUM,CONSTITUENCY_TITLE,LEN_CIRC
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,LOVI...",121
1,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE,17
2,AGNEBY-TIASSA,003,AZAGUIE COMMUNE ET SOUS-PREFECTURE,34
3,AGNEBY-TIASSA,004,"ANANGUIE, CECHI ET RUBINO, COMMUNES ET SOUS-PR...",55
4,AGNEBY-TIASSA,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES",46
...,...,...,...,...
207,WORODOUGOU,201,"KANI, COMMUNE ET SOUS-PREFECTURE",32
208,WORODOUGOU,202,"BOBI-DIARABANA, COMMUNE ET SOUS-PREFECTURE,SEG...",67
209,WORODOUGOU,203,"SEGUELA, COMMUNE",16
210,WORODOUGOU,204,"DUALLA ET MASSALA, COMMUNES ET SOUS-PREFECTURES",47


In [35]:
circs.describe().T

,count,mean,std,min,25%,50%,75%,max
LEN_CIRC,212.0,50.872642,23.645442,0.0,38.5,52.0,65.0,130.0


In [36]:
final_circs = circs[circs["LEN_CIRC"]>0][
    ["REGION", "CONSTITUENCY_NUM", "CONSTITUENCY_TITLE"]
]

final_circs.head()

,REGION,CONSTITUENCY_NUM,CONSTITUENCY_TITLE
0,AGNEBY-TIASSA,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIÉ,LOVI..."
1,AGNEBY-TIASSA,002,AGBOVILLE COMMUNE
2,AGNEBY-TIASSA,003,AZAGUIE COMMUNE ET SOUS-PREFECTURE
3,AGNEBY-TIASSA,004,"ANANGUIE, CECHI ET RUBINO, COMMUNES ET SOUS-PR..."
4,AGNEBY-TIASSA,005,"GOMON ET SIKENSI, COMMUNES ET SOUS-PREFECTURES"


In [37]:
final_circs.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "constituencies.parquet"), index=False)

### Entity aliases for constituencies

In [38]:

# Administrative stop words
CIV_STOP_WORDS = {
    "COMMUNES", "ET", "SOUS", "PREFECTURES", "PREFECTURE", "SOUS-PREFECTURES", 
    "SOUS-PREFECTURE", "COMMUNE", "VILLE", "DE", "LA", "LE", "LES"
}

def generate_aliases(constituency_num, full_title):
    full_title = normalize_text(
        full_title, 
        lowercase=False, 
        remove_punctuation=False
    )
    clean_title = re.sub(r'[,]', ' ', full_title.upper())
    words = clean_title.split()
    aliases = [w for w in words if w not in CIV_STOP_WORDS and len(w) > 2]
    
    return [(alias, constituency_num, full_title) for alias in aliases]


In [39]:
all_alias_records = []

for idx, row in circs.iterrows():
    _, circ_num, circ_title, _ = row.values
    
    aliases = generate_aliases(circ_num, circ_title)
    
    all_alias_records.extend(aliases)

In [40]:
entity_alias_df = pd.DataFrame(
    all_alias_records, 
    columns=['SEARCH_TERM', 'CONSTITUENCY_NUM', 'FORMAL_CONSTITUENCY_TITLE']
)

entity_alias_df.head()

,SEARCH_TERM,CONSTITUENCY_NUM,FORMAL_CONSTITUENCY_TITLE
0,ABOUDE,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIE,LOVI..."
1,ATTOBROU,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIE,LOVI..."
2,GUESSIGUIE,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIE,LOVI..."
3,GRAND-MORIE,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIE,LOVI..."
4,LOVIGUIE,001,"ABOUDE, ATTOBROU, GUESSIGUIE, GRAND-MORIE,LOVI..."


In [41]:
entity_alias_df.shape

(556, 3)

In [42]:
entity_alias_df[entity_alias_df.SEARCH_TERM == "TIAPOUM"]

,SEARCH_TERM,CONSTITUENCY_NUM,FORMAL_CONSTITUENCY_TITLE
493,TIAPOUM,184,"NOE, NOUAMOU ET TIAPOUM, COMMUNES ET SOUS-PREF..."


In [43]:
entity_alias_df.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "entity_alias.parquet"), index=False)

### Party

In [44]:
party_data = pd.DataFrame(df["PARTY_NAME"].unique(), columns=["PARTY_NAME"])
party_data["PARTY_NAME"] = party_data["PARTY_NAME"].apply(lambda n: n.replace(' ', ''))

party_data.head()

,PARTY_NAME
0,INDEPENDANT
1,RHDP
2,ADCI
3,FPI
4,MGC


In [45]:
party_data["PARTY_NAME"] = party_data["PARTY_NAME"].apply(lambda p: normalize_text(p, remove_punctuation=True, lowercase=False))
party_data["PARTY_NORMALIZED"] = party_data["PARTY_NAME"].apply(lambda p: normalize_text(p, remove_punctuation=True))

In [46]:
party_data

,PARTY_NAME,PARTY_NORMALIZED
0,INDEPENDANT,independant
1,RHDP,rhdp
2,ADCI,adci
3,FPI,fpi
4,MGC,mgc
5,PDCI-RDA,pdci-rda
6,CNPCIN,cnpcin
7,ADCI-GP-PAIX-VALEUR,adci-gp-paix-valeur
8,GJPA-CI,gjpa-ci
9,CNJB-ADO,cnjb-ado


In [47]:
party_data.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "parties.parquet"), index=False)

### Candidates data

In [48]:
candidates = df[
    ["PARTY_NAME","CANDIDATE_NAME", "CONSTITUENCY_NUM"]
].drop_duplicates().reset_index(drop=True)

candidates["PARTY_NAME"] = candidates["PARTY_NAME"].apply(lambda p: normalize_text(p, remove_punctuation=True, lowercase=False))
candidates["CANDIDATE_NAME"] = candidates["CANDIDATE_NAME"].apply(lambda p: normalize_text(p, remove_punctuation=True, lowercase=False))

candidates

,PARTY_NAME,CANDIDATE_NAME,CONSTITUENCY_NUM
0,INDEPENDANT,KOTO EHOU SOPIE,001
1,RHDP,KOFFI AKA CHARLES,001
2,INDEPENDANT,YEPIE KOUASSI THEODORE,001
3,INDEPENDANT,TCHIMOU GNAMON BERTRAND,001
4,INDEPENDANT,BOKA BOKA MAURICE,001
...,...,...,...
1119,RHDP,MEITE ABOULAYE,205
1120,INDEPENDANT,KONE LOSSENI,205
1121,INDEPENDANT,TRAORE SEKOU,205
1122,RDP,FOFANA SIAKA,205


In [49]:
candidates.CANDIDATE_NAME.unique()

array(['KOTO EHOU SOPIE', 'KOFFI AKA CHARLES', 'YEPIE KOUASSI THEODORE',
       ..., 'TRAORE SEKOU', 'FOFANA SIAKA', 'TRAORE MATAGALY KARIDJA'],
      shape=(1064,), dtype=object)

In [50]:
candidates = candidates.drop_duplicates(subset=['CANDIDATE_NAME'], keep='first', inplace=False)

candidates

,PARTY_NAME,CANDIDATE_NAME,CONSTITUENCY_NUM
0,INDEPENDANT,KOTO EHOU SOPIE,001
1,RHDP,KOFFI AKA CHARLES,001
2,INDEPENDANT,YEPIE KOUASSI THEODORE,001
3,INDEPENDANT,TCHIMOU GNAMON BERTRAND,001
4,INDEPENDANT,BOKA BOKA MAURICE,001
...,...,...,...
1119,RHDP,MEITE ABOULAYE,205
1120,INDEPENDANT,KONE LOSSENI,205
1121,INDEPENDANT,TRAORE SEKOU,205
1122,RDP,FOFANA SIAKA,205


In [51]:
candidates.to_parquet(osp.join(CFG.PROCESSED_DATA_DIR, "candidates.parquet"), index=False)